# Adım 7: Dashboard ve Görselleştirme
**Kişi 3 sorumluluğu** — `feature/ml-dashboard` branch

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os

GOLD_PATH    = './delta_lake/gold'
FEATURE_PATH = './delta_lake/features'
MLFLOW_DIR   = './mlflow_data'
PLOTS_DIR    = './plots'
os.makedirs(PLOTS_DIR, exist_ok=True)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})

FEATURE_COLS = [
    'temp_range', 'month', 'season_num', 'rolling_avg_7',
    'is_extreme_temp', 'prcp_category',
    'avg_wind_speed_kmh', 'avg_sea_level_pres_hpa', 'sunshine_total_min',
]
FEATURE_LABELS = [
    'Sicaklik Farki', 'Ay', 'Mevsim', '7g Hareketli Ort.',
    'Asiri Sicaklik', 'Yagis Kategorisi',
    'Ruzgar Hizi', 'Deniz Seviyesi Bas.', 'Gunes Suresi',
]

def create_spark():
    return (
        SparkSession.builder
        .appName('ClimateDashboard')
        .master('local[*]')
        .config('spark.sql.extensions',
                'io.delta.sql.DeltaSparkSessionExtension')
        .config('spark.sql.catalog.spark_catalog',
                'org.apache.spark.sql.delta.catalog.DeltaCatalog')
        .config('spark.jars.packages',
                'io.delta:delta-core_2.12:2.4.0')
        .getOrCreate()
    )

spark = create_spark()
spark.sparkContext.setLogLevel('WARN')
print('[Dashboard] Spark session hazir.')

In [ ]:
# D01: 5 modelin RMSE/MAE/R2 karsilastirmasi
df_cmp = pd.read_csv(f'{MLFLOW_DIR}/model_comparison.csv')
metrics = ['rmse', 'mae', 'r2']
labels  = ['RMSE', 'MAE', 'R2']
x = range(len(df_cmp))
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('5 Modelin Performans Karsilastirmasi', fontsize=14, fontweight='bold')
colors = sns.color_palette('Set2', 5)
for i, (metric, label) in enumerate(zip(metrics, labels)):
    bars = axes[i].bar(x, df_cmp[metric], color=colors)
    axes[i].set_title(label)
    axes[i].set_xticks(list(x))
    axes[i].set_xticklabels(df_cmp['model'], rotation=20, ha='right', fontsize=9)
    axes[i].bar_label(bars, fmt='%.3f', padding=2, fontsize=8)
    if metric == 'r2': axes[i].set_ylim(0, 1.1)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/D01_model_comparison.png'); plt.show()
print('[Dashboard 1] Model karsilastirma grafigi kaydedildi.')

In [ ]:
# D02: Feature importance
with open(f'{MLFLOW_DIR}/feature_importance.json') as f:
    fi_all = json.load(f)
model_name = 'RandomForest' if 'RandomForest' in fi_all else list(fi_all.keys())[0]
fi = fi_all[model_name]
df_fi = pd.DataFrame({'feature': FEATURE_LABELS,
                      'importance': [fi.get(c, 0) for c in FEATURE_COLS]})
df_fi = df_fi.sort_values('importance')
fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(df_fi['feature'], df_fi['importance'],
               color=sns.color_palette('Blues_r', len(df_fi)))
ax.set_title(f'Ozellik Onemi — {model_name}', fontsize=13, fontweight='bold')
ax.set_xlabel('Onem Skoru')
ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=9)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/D02_feature_importance.png'); plt.show()
print('[Dashboard 2] Feature importance grafigi kaydedildi.')

In [ ]:
# D03: Aylik sicaklik trend
ay_isimleri = ['Oca','Sub','Mar','Nis','May','Haz','Tem','Agu','Eyl','Eki','Kas','Ara']
df_spark = spark.read.format('delta').load(GOLD_PATH)
pdf = (df_spark.groupBy('month').agg(avg('avg_temp_c').alias('ort'))
       .orderBy('month').toPandas())
pdf['ay'] = pdf['month'].apply(lambda x: ay_isimleri[int(x) - 1])
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(pdf['ay'], pdf['ort'], marker='o', linewidth=2.5, color='#e63946', markersize=8)
ax.fill_between(range(len(pdf)), pdf['ort'], alpha=0.15, color='#e63946')
ax.set_title('Aylik Ortalama Kuresel Sicaklik Trendi', fontsize=13, fontweight='bold')
ax.set_xlabel('Ay'); ax.set_ylabel('Ortalama Sicaklik (°C)')
ax.set_xticks(range(len(pdf))); ax.set_xticklabels(pdf['ay'])
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/D03_monthly_trend.png'); plt.show()
print('[Dashboard 3] Aylik trend grafigi kaydedildi.')

In [ ]:
# D04: Sicaklik dagilim histogram
pdf = df_spark.select('avg_temp_c').dropna().limit(100000).toPandas()
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(pdf['avg_temp_c'], bins=80, color='#457b9d', edgecolor='white', alpha=0.85)
ax.axvline(pdf['avg_temp_c'].mean(), color='#e63946', linestyle='--', linewidth=2,
           label=f"Ortalama: {pdf['avg_temp_c'].mean():.1f}°C")
ax.set_title('Gunluk Ortalama Sicaklik Dagilimi', fontsize=13, fontweight='bold')
ax.set_xlabel('Sicaklik (°C)'); ax.set_ylabel('Frekans'); ax.legend()
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/D04_temp_histogram.png'); plt.show()
print('[Dashboard 4] Sicaklik histogram kaydedildi.')

## Dashboard — Görsel 5-8

In [ ]:
# D05: Gercek vs Tahmin scatter
from pyspark.ml import PipelineModel
model_path = './mlflow_data/best_model'
if os.path.exists(model_path):
    model = PipelineModel.load(model_path)
    df_feat = spark.read.format('delta').load(FEATURE_PATH).dropna()
    preds = model.transform(df_feat).select(
        col('avg_temp_c').alias('gercek'),
        col('prediction').alias('tahmin')
    ).limit(5000).toPandas()
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(preds['gercek'], preds['tahmin'], alpha=0.3, s=6, color='#2a9d8f')
    lims = [preds['gercek'].min(), preds['gercek'].max()]
    ax.plot(lims, lims, 'r--', linewidth=1.5, label='Ideal (y=x)')
    ax.set_title('Gercek vs Tahmin Sicaklik', fontsize=13, fontweight='bold')
    ax.set_xlabel('Gercek Sicaklik (°C)'); ax.set_ylabel('Tahmin Edilen Sicaklik (°C)')
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'{PLOTS_DIR}/D05_actual_vs_predicted.png'); plt.show()
    print('[Dashboard 5] Gercek vs tahmin scatter kaydedildi.')
else:
    print('[Dashboard 5] Model bulunamadi, atlaniyor.')

In [ ]:
# D06: Residual dagilim
if os.path.exists(model_path):
    preds_r = model.transform(df_feat).select(
        (col('prediction') - col('avg_temp_c')).alias('residual')
    ).limit(5000).toPandas()
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(preds_r['residual'], bins=60, color='#e9c46a', edgecolor='white', alpha=0.9)
    ax.axvline(0, color='#e63946', linestyle='--', linewidth=2)
    ax.set_title('Residual (Artik) Dagilimi', fontsize=13, fontweight='bold')
    ax.set_xlabel('Residual (Tahmin - Gercek)'); ax.set_ylabel('Frekans')
    plt.tight_layout()
    plt.savefig(f'{PLOTS_DIR}/D06_residuals.png'); plt.show()
    print('[Dashboard 6] Residual dagilim grafigi kaydedildi.')
else:
    print('[Dashboard 6] Model bulunamadi, atlaniyor.')

In [ ]:
# D07: Top 20 sehir ortalama sicaklik
pdf = (df_spark.groupBy('city_name').agg(avg('avg_temp_c').alias('ort'))
       .orderBy(col('ort').desc()).limit(20).toPandas())
fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(pdf['city_name'][::-1], pdf['ort'][::-1],
               color=sns.color_palette('YlOrRd', 20))
ax.set_title('En Sicak 20 Sehir — Ortalama Sicaklik', fontsize=13, fontweight='bold')
ax.set_xlabel('Ortalama Sicaklik (°C)')
ax.bar_label(bars, fmt='%.1f°C', padding=3, fontsize=9)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/D07_top20_cities.png'); plt.show()
print('[Dashboard 7] Top 20 sehir grafigi kaydedildi.')

In [ ]:
# D08: Korelasyon heatmap
num_cols = ['avg_temp_c', 'temp_range', 'rolling_avg_7',
            'avg_wind_speed_kmh', 'avg_sea_level_pres_hpa',
            'sunshine_total_min', 'prcp_category']
df_feat2 = spark.read.format('delta').load(FEATURE_PATH)
pdf = df_feat2.select(num_cols).dropna().limit(30000).toPandas()
corr = pdf.corr()
labels = ['Ort. Sicaklik', 'Sicaklik Farki', '7g Hareketli',
          'Ruzgar Hizi', 'Deniz Bas.', 'Gunes Suresi', 'Yagis Kat.']
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=ax,
            xticklabels=labels, yticklabels=labels)
ax.set_title('Ozellikler Arasi Korelasyon Matrisi', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/D08_correlation_heatmap.png'); plt.show()
print('[Dashboard 8] Korelasyon heatmap kaydedildi.')

print(f"\nTum dashboard gorselleri '{PLOTS_DIR}' klasorune kaydedildi.")
spark.stop()